# ---SCRAPING---

homedy.com

In [ ]:
import os
import subprocess
from selenium import webdriver
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.chrome.service import Service
from webdriver_manager.chrome import ChromeDriverManager
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
import pandas as pd
import time
import random
import re

# --- CẤU HÌNH ---
FILENAME = 'data_homedy_full.csv' # Tên file CSV của bạn
BASE_URL = "https://homedy.com/cho-thue-nha-tro-phong-tro-tp-ho-chi-minh"
START_PAGE = 41
END_PAGE = 104

# --- KHỞI TẠO DRIVER ---
def get_driver():
    try:
        subprocess.run(['pkill', '-9', 'chrome'], stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
    except: pass
    
    options = Options()
    options.add_argument('--headless=new')
    options.add_argument('--no-sandbox')
    options.add_argument('--disable-dev-shm-usage')
    options.add_argument('--disable-blink-features=AutomationControlled')
    options.add_argument("user-agent=Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/124.0.0.0 Safari/537.36")
    
    service = Service(ChromeDriverManager().install())
    driver = webdriver.Chrome(service=service, options=options)
    driver.set_page_load_timeout(60)
    return driver

# --- CÁC HÀM XỬ LÝ ---
def clean_price(text):
    if not text: return 0
    try:
        text = text.lower().replace(',', '.')
        nums = re.findall(r"[-+]?\d*\.\d+|\d+", text)
        if not nums: return 0
        val = float(nums[0])
        if 'triệu' in text: return int(val * 1000000)
        if 'tỷ' in text: return int(val * 1000000000)
        if 'nghìn' in text: return int(val * 1000)
        return int(val)
    except: return 0

def clean_area(text):
    if not text: return 0.0
    try:
        text = text.lower().replace(',', '.')
        nums = re.findall(r"[-+]?\d*\.\d+|\d+", text)
        return float(nums[0]) if nums else 0.0
    except: return 0.0

def clean_district(text):
    if not text or text == "N/A": return "N/A"
    parts = text.split(',')
    if len(parts) >= 2:
        return parts[-2].strip()
    return text.strip()

def get_text(element, selector):
    try:
        return element.find_element(By.CSS_SELECTOR, selector).text.strip()
    except:
        return "N/A"

def get_full_description(driver, link):
    """Vào trang chi tiết lấy mô tả full"""
    try:
        driver.get(link)
        try:
            WebDriverWait(driver, 5).until(EC.presence_of_element_located((By.CSS_SELECTOR, "h1")))
            # Các class có thể chứa mô tả
            selectors = [".description", ".product-description", "#description-content", ".description-content"]
            for sel in selectors:
                try:
                    elem = driver.find_element(By.CSS_SELECTOR, sel)
                    if elem.is_displayed():
                        return elem.text.strip().replace('\n', ' ')
                except: continue
        except: pass
        return "N/A"
    except: return "Error"

# --- CHẠY CHÍNH ---
def run_scraper():
    driver = None
    current_index = 0
    batch_data = [] # Bộ nhớ tạm chứa dữ liệu của 1 trang
    
    # Kiểm tra file cũ
    if os.path.exists(FILENAME):
        try:
            df_check = pd.read_csv(FILENAME)
            if not df_check.empty:
                current_index = df_check['index'].max() + 1
                print(f"📂 Tim thay file cu '{FILENAME}'. Tiep tuc tu Index: {current_index}")
        except: pass

    try:
        driver = get_driver()
        print(f"🚀 Bat dau cao du lieu tu trang {START_PAGE} den {END_PAGE}...")

        for page in range(START_PAGE, END_PAGE + 1):
            url = f"{BASE_URL}/p{page}"
            print(f"\n--- DANG XU LY TRANG {page} ---")
            
            try:
                driver.get(url)
                WebDriverWait(driver, 15).until(EC.presence_of_element_located((By.CSS_SELECTOR, ".product-item")))
            except:
                print("   ! Loi tai trang hoac het tin.")
                continue

            # 1. Lấy thông tin cơ bản từ list
            items = driver.find_elements(By.CSS_SELECTOR, ".product-item")
            print(f"   -> Tim thay {len(items)} tin. Dang lay Link & Info co ban...")
            
            temp_list = []
            for item in items:
                try:
                    # Selector chuẩn cho Homedy: h3 a
                    try:
                        title_el = item.find_element(By.CSS_SELECTOR, "h3 a")
                    except:
                        title_el = item.find_element(By.CSS_SELECTOR, ".title a")
                        
                    title = title_el.text.strip()
                    link = title_el.get_attribute('href')
                    if link and not link.startswith('http'):
                        link = "https://homedy.com" + link
                    
                    price_raw = get_text(item, ".price")
                    area_raw = get_text(item, ".acreage")
                    district_raw = get_text(item, ".address")
                    
                    temp_list.append({
                        "link": link,
                        "title": title,
                        "price_raw": price_raw,
                        "area_raw": area_raw,
                        "district_raw": district_raw
                    })
                except: continue
            
            # 2. Vào chi tiết từng tin để lấy Full Desc
            batch_data = [] # Reset mỗi khi sang trang mới
            print(f"   -> Bat dau vao chi tiet {len(temp_list)} tin...")
            
            for i, temp in enumerate(temp_list):
                full_desc = get_full_description(driver, temp['link'])
                
                row = {
                    'index': current_index,
                    'url': temp['link'],
                    'title': temp['title'],
                    'price': clean_price(temp['price_raw']),
                    'area': clean_area(temp['area_raw']),
                    'district': clean_district(temp['district_raw']),
                    'description': full_desc, # FULL TEXT
                    'source': 'homedy.com'
                }
                batch_data.append(row)
                current_index += 1
                
                print(f"     + [{i+1}/{len(temp_list)}] {temp['title'][:30]}... | Desc: {len(full_desc)} chars")
                time.sleep(random.uniform(1.5, 2.5)) # Nghỉ ngắn

            # 3. LOGIC LƯU FILE (LƯU NGAY SAU MỖI TRANG)
            if batch_data:
                df = pd.DataFrame(batch_data)
                
                # Nếu file chưa tồn tại hoặc rỗng thì ghi header
                hdr = not os.path.exists(FILENAME) or os.path.getsize(FILENAME) == 0
                
                df.to_csv(FILENAME, mode='a', header=hdr, index=False, encoding='utf-8-sig')
                print(f"\n💾 ✅ Đã thêm {len(batch_data)} tin mới vào file: {FILENAME}")
            else:
                print(f"\n⚠️ Trang {page} khong co du lieu de luu.")
            
            time.sleep(3)

    except Exception as e:
        print(f"❌ LOI CHUONG TRINH: {e}")
        if batch_data:
            print("💾 Dang luu du lieu con lai truoc khi thoat...")
            df = pd.DataFrame(batch_data)
            df.to_csv(FILENAME, mode='a', header=False, index=False, encoding='utf-8-sig')

    finally:
        if driver: driver.quit()
        print("\n🎉 CHUONG TRINH KET THUC!")

if __name__ == "__main__":
    run_scraper()

merge 4 file

# ---TRÍCH XUẤT DỮ LIỆU---

In [ ]:
import pandas as pd
import re

# ---------------------------------------------------------
# CẤU HÌNH INPUT
# ---------------------------------------------------------
input_files = [
    'chotot_clear.csv',
    
]
output_file = 'final_data_clean_v4.csv'

# ---------------------------------------------------------
# 1. HÀM XỬ LÝ ĐỊA CHỈ & LỌC SỐ NHÀ (BẢN MỚI NHẤT)
# ---------------------------------------------------------
def clean_text(text):
    return str(text).strip() if pd.notnull(text) else ""

def remove_house_number(street):
    if not street: return ""
    street = street.strip()
    
    # BƯỚC 1: Xử lý trường hợp có dấu phẩy (VD: "16/2, 44")
    if ',' in street:
        parts = [p.strip() for p in street.split(',')]
        # Nếu phần đầu tiên là số nhà, bỏ nó đi
        if len(parts) > 1 and re.match(r'^(số|no\.|lô|căn)?\s*\d+[\w\/\.-]*$', parts[0], re.IGNORECASE):
            street = ", ".join(parts[1:])

    # BƯỚC 2: Xử lý dạng không dấu phẩy (VD: "123 Nguyễn Xí")
    street = re.sub(r'^(số|no\.|lô|căn)\s+\d+[\w\/-]*\s*', '', street, flags=re.IGNORECASE)
    street = re.sub(r'^\d+[\w\/-]*\s+', '', street)

    # BƯỚC 3: Làm sạch từ khóa thừa
    street = re.sub(r'^(đường|phố|street)\s+', '', street, flags=re.IGNORECASE)
    street = street.strip(" ,.-")
    
    return street.strip().title()

def extract_address_components(addr):
    addr = clean_text(addr)
    result = {'extracted_street': '', 'extracted_ward': '', 'extracted_district': ''}
    if not addr: return pd.Series(result)

    parts = [p.strip() for p in re.split(r'[,|-]', addr)]
    
    # Tìm Quận
    district_idx = -1
    for i in range(len(parts) - 1, -1, -1):
        p = parts[i].lower()
        if re.search(r'^(quận|q\.|q|huyện|thị xã|tp|thành phố)\s*.+', p) or 'thủ đức' in p:
            if 'hồ chí minh' in p or p == 'hcm': continue
            result['extracted_district'] = parts[i].title()
            district_idx = i
            break
            
    # Tìm Phường
    ward_idx = -1
    if district_idx > 0:
        prev = parts[district_idx - 1].lower()
        if re.search(r'^(phường|p\.|p|xã)\s*.+', prev) or prev.isdigit():
             result['extracted_ward'] = parts[district_idx - 1].title()
             ward_idx = district_idx - 1
    elif district_idx == -1:
        for i, p in enumerate(parts):
            if re.search(r'^(phường|p\.|p|xã)\s*.+', p.lower()):
                result['extracted_ward'] = parts[i].title()
                ward_idx = i
                break

    # Tìm Đường (Và xóa số nhà)
    cut = ward_idx if ward_idx != -1 else district_idx
    raw_street = ""
    if cut > 0: 
        raw_street = ", ".join(parts[:cut])
    elif cut == -1: 
        raw_street = addr 
        
    result['extracted_street'] = remove_house_number(raw_street)

    return pd.Series(result)

# ---------------------------------------------------------
# 2. HÀM RÚT TRÍCH FEATURE (ĐÃ BỎ PHONE, PETS)
# ---------------------------------------------------------
def extract_full_features(desc):
    text = clean_text(desc).lower()
    
    def find_price(keywords):
        kw_pattern = '|'.join(keywords)
        match = re.search(fr'({kw_pattern})[\s:]*(\d+[.,\d]*)', text)
        if match:
            num_str = match.group(2).replace('.', '').replace(',', '')
            val = float(num_str)
            return val * 1000 if val < 500 else val
        return -1 

    data = {}

    # --- NHÓM GIÁ DỊCH VỤ ---
    data['deposit']         = find_price(['cọc'])
    data['electricity_price'] = find_price(['điện', 'dien'])
    data['water_price']     = find_price(['nước', 'nuoc'])
    data['internet_price']  = find_price(['wifi', 'net', 'mạng'])
    data['parking_price']   = find_price(['xe', 'giữ xe', 'để xe'])
    data['cleaning_price']  = find_price(['rác', 'vệ sinh'])

    # --- NHÓM TIỆN ÍCH & NỘI THẤT (Đã bỏ pets_allowed) ---
    keywords_map = {
        'has_balcony':     ['ban công', 'balcony', 'cửa sổ lớn'],
        'has_window':      ['cửa sổ', 'thoáng'],
        'has_mezzanine':   ['gác', 'duplex', 'lửng'],
        'has_elevator':    ['thang máy', 'thang may', 'elevator'],
        'has_parking':     ['bãi xe', 'hầm xe', 'để xe', 'giữ xe'],
        'has_ac':          ['máy lạnh', 'điều hòa', 'air con'],
        'has_fan':         ['quạt'],
        'has_ceiling_fan': ['quạt trần'],
        'has_water_heater':['nước nóng', 'nóng lạnh', 'heater'],
        'has_kitchen':     ['kệ bếp', 'bếp', 'nấu ăn'],
        'has_fridge':      ['tủ lạnh'],
        'has_washing_machine': ['máy giặt', 'giặt chung'],
        'has_bed':         ['giường', 'nệm'],
        'has_wardrobe':    ['tủ quần áo', 'tủ đồ'],
        'is_furnished':    ['full nội thất', 'đầy đủ tiện nghi'],
        'has_fingerprint': ['vân tay', 'khoá từ'],
        'has_camera':      ['camera', 'an ninh'],
        'is_free_time':    ['tự do', 'chìa khóa riêng', '24/24', 'không chung chủ'],
    }

    for col, kws in keywords_map.items():
        data[col] = any(kw in text for kw in kws)

    # --- PHÂN LOẠI ---
    # WC
    if any(k in text for k in ['khép kín', 'wc riêng', 'toilet riêng']):
        data['wc_type'] = 'private'
    elif any(k in text for k in ['wc chung', 'vệ sinh chung']):
        data['wc_type'] = 'shared'
    else:
        data['wc_type'] = 'private' if 'khép kín' in text else 'unknown'

    # Room Type
    if any(k in text for k in ['ktx', 'ký túc xá', 'giường tầng', 'dorm']):
        data['room_type'] = 'ktx'
    elif any(k in text for k in ['sleepbox', 'hộp ngủ']):
        data['room_type'] = 'sleepbox'
    else:
        data['room_type'] = 'room'

    # Đã xoá logic trích xuất contact_phone tại đây
    
    return pd.Series(data)

# ---------------------------------------------------------
# 3. CHẠY XỬ LÝ
# ---------------------------------------------------------
all_dfs = []
print("Bắt đầu xử lý...")

for f in input_files:
    try:
        print(f"-> Đang đọc: {f}")
        df = pd.read_csv(f)
        
        # 1. Tách địa chỉ
        if 'address' in df.columns:
            addr_cols = df['address'].apply(extract_address_components)
            df = pd.concat([df, addr_cols], axis=1)
            
        # 2. Rút trích Description
        if 'description' in df.columns:
            desc_cols = df['description'].apply(extract_full_features)
            df = pd.concat([df, desc_cols], axis=1)
            
        all_dfs.append(df)
    except Exception as e:
        print(f"Lỗi file {f}: {e}")

if all_dfs:
    final_df = pd.concat(all_dfs, ignore_index=True)
    
    # --- BƯỚC CUỐI: XOÁ CỘT KHÔNG CẦN THIẾT ---
    # Xoá cột 'index' (nếu có trong file gốc) và 'contact_phone' (để chắc chắn)
    cols_to_drop = ['index', 'contact_phone', 'Unnamed: 0']
    final_df.drop(columns=cols_to_drop, errors='ignore', inplace=True)
    
    final_df.to_csv(output_file, index=False, encoding='utf-8-sig')
    print(f"\nĐã xuất file thành công: {output_file}")
    print(f"Số dòng: {len(final_df)}")
    
    # Kiểm tra lại các cột còn lại
    print("\nDanh sách các cột trong file kết quả:")
    print(list(final_df.columns))

# ---LÀM SẠCH DỮ LIỆU VÀ EDA

PHẦN 1: KHỞI TẠO MÔI TRƯỜNG
Trước khi bắt đầu, chúng ta cần nhập các thư viện cần thiết:
* **Pandas & Numpy:** Để thao tác và xử lý dữ liệu dạng bảng và số học.
* **Matplotlib & Seaborn:** Để vẽ biểu đồ trực quan hóa dữ liệu (EDA).
* **Re (Regex):** Để xử lý chuỗi ký tự và trích xuất thông tin từ địa chỉ.

Cấu hình `warnings.filterwarnings('ignore')` giúp notebook gọn gàng hơn bằng cách ẩn các cảnh báo không quan trọng.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import re
import warnings

# Cấu hình hiển thị
pd.set_option('display.max_columns', None) # Hiển thị tất cả cột
pd.set_option('display.max_colwidth', 100) # Hiển thị nội dung dài
sns.set_style("whitegrid") # Giao diện biểu đồ sáng sủa
warnings.filterwarnings('ignore') # Tắt cảnh báo

PHẦN 2: ĐỌC DỮ LIỆU (DATA LOADING)
Chúng ta sẽ đọc file `data_homedy_full.csv` vào DataFrame của Pandas.
Bước này cũng kiểm tra xem file có tồn tại hay không để tránh lỗi chương trình bị dừng đột ngột.

In [ ]:
try:
    df = pd.read_csv('data_homedy_full.csv')
    pd.read_csv()
    print(f"Đọc file thành công! Kích thước dữ liệu: {df.shape}")
    print("-" * 30)
    print("3 dòng đầu tiên của dữ liệu:")
    display(df.head(3))
except FileNotFoundError:
    print("Lỗi: Không tìm thấy file 'data_homedy_full.csv'. Kiểm tra đường dẫn.")

PHẦN 3: TIỀN XỬ LÝ & LÀM SẠCH (DATA CLEANING)
Theo nguyên tắc **"Garbage In, Garbage Out"** (Rác vào thì Rác ra), đây là bước quan trọng nhất.

Chúng ta sẽ thực hiện 4 tác vụ:
1.  **Xóa cột rác (Garbage Collection):** Loại bỏ `url`, `source`, `Unnamed: 0` vì chúng không giúp mô hình dự đoán giá nhà.
2.  **Xử lý trùng lặp (De-duplication):** Dữ liệu cào web thường bị lặp lại do một tin được đăng nhiều lần. Ta sẽ xóa các dòng trùng nhau hoàn toàn.
3.  **Ép kiểu (Type Casting):** Chuyển đổi `price` và `area` từ dạng chuỗi sang dạng số. Các giá trị lỗi (vd: "Thỏa thuận") sẽ bị chuyển thành `NaN`.
4.  **Lọc nhiễu (Filtering):** Áp dụng kiến thức thực tế (Domain Knowledge):
    * Xóa các dòng bị `NaN`.
    * Loại bỏ tin có Giá < 500k hoặc Diện tích < 5m2 (thường là dữ liệu rác hoặc lỗi nhập liệu).

In [ ]:
# 1. Xóa cột rác
cols_to_drop = ['source', 'url']
if 'Unnamed: 0' in df.columns:
    cols_to_drop.append('Unnamed: 0')

df = df.drop(columns=cols_to_drop, errors='ignore')
print(f"-> Đã xóa các cột: {cols_to_drop}")

# 2. Xử lý trùng lặp
before = len(df)
df = df.drop_duplicates(subset=['title', 'price', 'area', 'address'])
print(f"-> Đã xóa {before - len(df)} dòng trùng lặp.")

# 3. Ép kiểu dữ liệu (errors='coerce' sẽ biến lỗi thành NaN)
df['price'] = pd.to_numeric(df['price'], errors='coerce')
df['area'] = pd.to_numeric(df['area'], errors='coerce')

# 4. Lọc nhiễu và xử lý NaN
df = df.dropna(subset=['price', 'area']) # Xóa dòng NaN
df = df[(df['price'] > 500000) & (df['area'] > 5)] # Lọc theo logic thực tế

print(f"Dữ liệu sạch còn lại: {len(df)} dòng.")

PHẦN 4: KỸ THUẬT VỚI BIẾN SỐ (NUMERICAL FEATURE ENGINEERING)
**Lý thuyết:** Dữ liệu tài chính (như Giá nhà) thường tuân theo **Phân phối đuôi dài (Power Law)** thay vì Phân phối chuẩn (Normal Distribution). Điều này làm cho các mô hình học máy hoạt động kém hiệu quả.

**Giải pháp:** Sử dụng kỹ thuật **Log Transformation**.
Hàm `np.log1p(x) = log(x + 1)` giúp "co" dải giá trị lớn lại, biến phân phối trở nên giống hình chuông (Bell curve) hơn, giúp mô hình học tốt hơn.

In [ ]:
# Tạo đặc trưng mới: Log của Giá
df['log_price'] = np.log1p(df['price'])

# Vẽ biểu đồ so sánh
fig, ax = plt.subplots(1, 2, figsize=(15, 5))

# Biểu đồ gốc (Thường bị lệch trái)
sns.histplot(df['price'], bins=50, kde=True, ax=ax[0], color='blue')
ax[0].set_title('Phân phối Giá Gốc (Bị lệch/Skewed)')
ax[0].ticklabel_format(style='plain', axis='x') # Tắt format khoa học 1e6

# Biểu đồ sau log (Dạng chuẩn)
sns.histplot(df['log_price'], bins=50, kde=True, ax=ax[1], color='green')
ax[1].set_title('Phân phối Giá sau Log Transform (Chuẩn hơn)')

plt.tight_layout()
plt.show()

PHẦN 5: XỬ LÝ BIẾN ĐỊA CHỈ (CATEGORICAL VARIABLES)
**Vấn đề:** Cột `address` chứa quá nhiều giá trị duy nhất (High Cardinality) do bao gồm cả tên đường, số nhà... Nếu đưa thẳng vào mô hình sẽ gây nhiễu (Overfitting).

**Giải pháp (Binning):** Chúng ta sẽ trích xuất thông tin cấp cao hơn là **Quận/Huyện**.
Chúng ta dùng **Regular Expression (Regex)** để bắt các cụm từ như "Quận 1", "Huyện Nhà Bè", "TP Thủ Đức".

In [ ]:
def extract_district(addr):
    """Hàm dùng Regex để bắt tên Quận/Huyện từ địa chỉ"""
    if not isinstance(addr, str):
        return "Khác"
    # Regex bắt: Quận X, Huyện Y, TP Thủ Đức (kể cả viết tắt TP.)
    match = re.search(r'(Quận\s[\w\d]+|Huyện\s[\w\d]+|Thành phố\sThủ Đức|TP\sThủ Đức|TP\.\sThủ Đức)', addr, re.IGNORECASE)
    if match:
        return match.group(0).title() # Viết hoa chữ cái đầu
    return "Khác"

# Áp dụng hàm
df['district'] = df['address'].apply(extract_district)

# Trực quan hóa Top 15 khu vực
plt.figure(figsize=(12, 6))
top_districts = df['district'].value_counts().head(15)
sns.barplot(x=top_districts.values, y=top_districts.index, palette='viridis')
plt.title('Top 15 Khu vực có lượng tin đăng nhiều nhất')
plt.xlabel('Số lượng tin')
plt.show()

PHẦN 6: KHAI THÁC DỮ LIỆU VĂN BẢN (TEXT MINING)
Thay vì xử lý NLP phức tạp, ta dùng phương pháp **"Dictionary-based filtering"** để tạo các biến nhị phân (0/1).

Chúng ta trả lời câu hỏi: *Tin đăng này có chứa từ khóa X không?*
* `has_furniture`: Có từ khóa "nội thất", "full option"... hay không?
* `has_aircon`: Có "máy lạnh", "điều hòa" hay không?
* `has_balcony`: Có "ban công", "cửa sổ" hay không?

Đây là các yếu tố tác động mạnh đến giá thuê.

In [ ]:
# Tạo các biến chỉ báo (Indicator Variables)
# 1. Nội thất
df['has_furniture'] = df['description'].str.contains('nội thất|full option|tiện nghi', case=False, na=False).astype(int)
# 2. Máy lạnh
df['has_aircon'] = df['description'].str.contains('máy lạnh|điều hòa', case=False, na=False).astype(int)
# 3. Ban công
df['has_balcony'] = df['description'].str.contains('ban công|cửa sổ|thoáng', case=False, na=False).astype(int)

# Vẽ biểu đồ Boxplot để xem tác động của nội thất đến giá
plt.figure(figsize=(8, 5))
sns.boxplot(x='has_furniture', y='log_price', data=df, palette="Set2")
plt.title('Tác động của Nội thất đến Giá thuê (Thang đo Log)')
plt.xticks([0, 1], ['Không có/Ko ghi', 'Có nội thất'])
plt.ylabel('Log Price')
plt.show()

# Kiểm tra con số thực tế
mean_yes = df[df['has_furniture']==1]['price'].mean()
mean_no = df[df['has_furniture']==0]['price'].mean()
print(f"💰 Giá trung bình CÓ nội thất: {mean_yes:,.0f} VND")
print(f"💰 Giá trung bình KHÔNG nội thất: {mean_no:,.0f} VND")